# Exercice 1 — Explorer et décrire un jeu de données — CORRECTION

**Difficulté :** Débutant  
**Durée estimée :** 20 min  

---

## Contexte

On reçoit un export CSV provenant d'un réseau de capteurs IoT installés dans un entrepôt industriel. Le fichier `capteurs_temperature.csv` contient des relevés de température et d'humidité enregistrés automatiquement toutes les 15 minutes.

Avant toute analyse, la première étape consiste toujours à comprendre ce que contient le jeu de données : sa structure, ses dimensions, ses types de données, et ses éventuelles anomalies.

**Objectif :** Réaliser une exploration complète d'un CSV et en extraire une description synthétique.

---

## Données

Fichier : `../data/capteurs_temperature.csv`

---

## Tâche 1 — Charger le fichier avec pandas

In [ ]:
import pandas as pd

df = pd.read_csv('../data/capteurs_temperature.csv')

print("Fichier chargé avec succès.")

**Explication :** `pd.read_csv()` lit un fichier CSV et le convertit en DataFrame pandas — la structure de données centrale de pandas. Le chemin `../data/` remonte d'un niveau depuis le dossier `exercices/` pour accéder au dossier `data/`.

## Tâche 2 — Afficher les 10 premières lignes

In [ ]:
df.head(10)

**Explication :** `.head(n)` affiche les n premières lignes. Par défaut (sans argument), il en affiche 5. Cette première lecture permet de repérer immédiatement les noms de colonnes, le format des données (dates, nombres, texte) et d'éventuelles valeurs suspectes.

## Tâche 3 — Combien de lignes et de colonnes ?

In [ ]:
lignes, colonnes = df.shape
print(f"Le DataFrame contient {lignes} lignes et {colonnes} colonnes.")

**Explication :** `.shape` retourne un tuple `(nb_lignes, nb_colonnes)`. C'est l'une des premières choses à vérifier : un jeu de données beaucoup plus petit qu'attendu peut indiquer un problème de chargement ou un filtre non souhaité.

## Tâche 4 — Quels sont les types de chaque colonne ?

In [ ]:
print(df.dtypes)
print()
# .info() donne encore plus d'informations en une seule commande
df.info()

**Explication :** `.dtypes` affiche le type pandas de chaque colonne. On remarque que `timestamp` est de type `object` (chaîne de caractères) et non `datetime` — il faudra le convertir pour toute opération temporelle. Les colonnes numériques (`float64`) sont correctement typées.

## Tâche 5 — Y a-t-il des valeurs manquantes ?

In [ ]:
valeurs_manquantes = df.isnull().sum()
print("Valeurs manquantes par colonne :")
print(valeurs_manquantes)
print()

# Calculer le pourcentage pour chaque colonne
pct_manquant = (df.isnull().sum() / len(df) * 100).round(2)
print("Pourcentage de valeurs manquantes :")
print(pct_manquant)

**Explication :** `.isnull()` crée un masque booléen (True là où la valeur est manquante), et `.sum()` compte les True par colonne. Diviser par `len(df)` donne la proportion. Dans ce jeu de données, aucune valeur n'est manquante — ce qui est rare en conditions réelles.

## Tâche 6 — Quels capteurs sont présents dans le jeu de données ?

In [ ]:
print("Capteurs présents :")
print(df['capteur_id'].unique())
print()

print("Nombre de relevés par capteur :")
print(df['capteur_id'].value_counts())

**Explication :** `.unique()` retourne les valeurs distinctes d'une colonne. `.value_counts()` retourne le décompte de chaque valeur, trié par fréquence décroissante. Vérifier que chaque capteur a le même nombre de relevés permet de détecter des lacunes dans la collecte.

## Tâche 7 — Quelle est la plage de dates couverte ?

In [ ]:
# Convertir la colonne timestamp en type datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])

date_debut = df['timestamp'].min()
date_fin = df['timestamp'].max()
duree = date_fin - date_debut

print(f"Début  : {date_debut}")
print(f"Fin    : {date_fin}")
print(f"Durée  : {duree}")

**Explication :** `pd.to_datetime()` convertit une colonne texte en type datetime, ce qui débloque toutes les opérations temporelles (filtrage par période, extraction du mois, calcul de durée). Sans cette conversion, `.min()` et `.max()` effectueraient un tri alphabétique sur du texte — ce qui donnerait des résultats incorrects.

## Tâche 8 — Statistiques descriptives

In [ ]:
# .describe() calcule automatiquement count, mean, std, min, quartiles et max
stats = df[['temperature_c', 'humidite_pct']].describe().round(2)
print(stats)

**Explication :** `.describe()` est le raccourci le plus rapide pour obtenir un profil statistique complet. On sélectionne uniquement les colonnes numériques pertinentes avec `df[['col1', 'col2']]`. `.round(2)` limite l'affichage à 2 décimales pour plus de lisibilité.

**Lecture des résultats :**
- La température moyenne est d'environ 23.5 °C, avec un écart-type de 3.7 °C
- L'humidité est stable autour de 49.8 %, avec peu de dispersion (std = 4.3)

## Tâche 9 — Identifier le capteur avec la température moyenne la plus élevée

In [ ]:
# Grouper par capteur et calculer la température moyenne
temp_moyenne_par_capteur = df.groupby('capteur_id')['temperature_c'].mean().round(2)
print("Température moyenne par capteur :")
print(temp_moyenne_par_capteur.sort_values(ascending=False))
print()

# Identifier le capteur le plus chaud
capteur_max = temp_moyenne_par_capteur.idxmax()
valeur_max = temp_moyenne_par_capteur.max()
print(f"Capteur le plus chaud : {capteur_max} ({valeur_max} °C en moyenne)")

**Explication :** `.groupby('colonne')` regroupe les lignes par valeur unique de la colonne. `.mean()` calcule ensuite la moyenne pour chaque groupe. `.idxmax()` retourne l'index (ici, l'identifiant du capteur) correspondant à la valeur maximale. Ce pattern `groupby → agrégation → tri` est fondamental en analyse de données.

## Tâche 10 — Bonus : repérer un comportement anormal

In [ ]:
# Définir les seuils d'anomalie : au-delà de ±2 écarts-types de la moyenne globale
moyenne = df['temperature_c'].mean()
ecart_type = df['temperature_c'].std()

seuil_haut = moyenne + 2 * ecart_type
seuil_bas = moyenne - 2 * ecart_type

print(f"Moyenne globale : {moyenne:.2f} °C")
print(f"Écart-type      : {ecart_type:.2f} °C")
print(f"Seuil haut      : {seuil_haut:.2f} °C")
print(f"Seuil bas       : {seuil_bas:.2f} °C")
print()

# Filtrer les relevés anomaux
anomalies = df[
    (df['temperature_c'] > seuil_haut) |
    (df['temperature_c'] < seuil_bas)
]

print(f"Relevés anomaux détectés : {len(anomalies)}")
print()

if len(anomalies) > 0:
    print("Répartition des anomalies par capteur :")
    print(anomalies.groupby('capteur_id')['temperature_c'].agg(['count', 'min', 'max']))

**Explication :** La règle des 2 écarts-types est une heuristique statistique classique : dans une distribution normale, environ 95 % des valeurs se trouvent à moins de 2 écarts-types de la moyenne. Les valeurs au-delà sont statistiquement rares et méritent vérification.

L'opérateur `|` dans pandas correspond à un OU logique pour filtrer les lignes. On peut aussi utiliser `df.query()` pour une syntaxe plus lisible sur des conditions complexes.

---

## Synthèse

Ce jeu de données contient **446 relevés** de **5 capteurs** (T01 à T05) couvrant la journée du **1er mars 2024**. Les données sont propres (pas de valeurs manquantes). La température varie entre 17 et 30 °C selon les capteurs et les heures, avec des niveaux d'humidité stables autour de 50 %.